# 01. Hello LLM

## What you'll learn

- **Making your first LLM API call** — sending a prompt and getting a response from a real model
- **The Chat Completions API** — endpoint, headers, request body, and response shape
- **Message roles** — `system`, `user`, and `assistant` — and how they shape LLM behavior
- **Temperature and sampling** — controlling randomness in model outputs
- **Multi-turn conversations** — sending conversation history so the LLM has context
- **Building a reusable `chat()` helper** — extracting boilerplate into a clean function

## Prerequisites

- [Appendix 01. Python Fundamentals](../appendix/01_python_fundamentals.ipynb) — variables, loops, control flow
- [Appendix 04. Strings and JSON](../appendix/04_strings_and_json.ipynb) — f-strings, `json.dumps`, parsing JSON
- [Appendix 05. Error Handling](../appendix/05_error_handling.ipynb) — `try`/`except` for graceful failure
- [Appendix 06. HTTP and APIs](../appendix/06_http_and_apis.ipynb) — HTTP methods, `httpx`, headers, status codes

## Why this matters for agents

Every AI agent starts with a single skill: **talking to an LLM**. Before you can build tool use, planning, memory, or multi-agent systems, you need to know how to send a prompt to a model and get a response back. This notebook teaches that foundational skill from scratch — no frameworks, no abstractions, just raw HTTP calls to a real LLM. By the end, you’ll have a working `chat()` function that every subsequent notebook builds on.

> **Further reading:**
> - [OpenRouter Quickstart](https://openrouter.ai/docs/quickstart) — official docs for the API we use throughout this repo
> - [OpenAI Chat Completions API reference](https://platform.openai.com/docs/api-reference/chat) — OpenRouter follows this same format
> - [Andrej Karpathy — Intro to Large Language Models (video)](https://www.youtube.com/watch?v=zjkBMFhNj_g) — excellent 1-hour overview of what LLMs are and how they work

---
## 1. Setup

Before we can talk to an LLM, we need an API key. We use [OpenRouter](https://openrouter.ai/) — a single API that gives access to many models (including free ones). Your key lives in a `.env` file at the project root.

If you haven’t set it up yet:
1. Copy `.env.example` to `.env` in the repo root
2. Sign up at [openrouter.ai](https://openrouter.ai/) and grab your API key
3. Paste it into `.env`: `OPENROUTER_API_KEY=sk-or-v1-...`

> **Note:** If the free model (`meta-llama/llama-3.1-8b-instruct:free`) is unavailable or rate-limited, try other free models on OpenRouter such as `mistralai/mistral-7b-instruct:free` or `google/gemma-2-9b-it:free`. Check [openrouter.ai/models](https://openrouter.ai/models) and filter by "Free" for the latest options.

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Retrieve the API key
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if OPENROUTER_API_KEY:
    print(f"API key loaded: {OPENROUTER_API_KEY[:12]}...")
else:
    print("WARNING: OPENROUTER_API_KEY not found!")
    print("Copy .env.example to .env and add your key before continuing.")

In [ ]:
# Constants we'll use throughout this notebook
API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "meta-llama/llama-3.1-8b-instruct:free"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

print(f"Endpoint: {API_URL}")
print(f"Model:    {MODEL}")

---
## 2. The Chat Completions API

OpenRouter (and OpenAI, and most LLM providers) use the **Chat Completions** format. You send a list of messages — each with a `role` and `content` — and the model returns a response.

### Request body structure

```json
{
  "model": "meta-llama/llama-3.1-8b-instruct:free",
  "messages": [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is Python?"}
  ],
  "temperature": 0.7
}
```

### Response shape

```json
{
  "choices": [
    {
      "message": {
        "role": "assistant",
        "content": "Python is a high-level programming language..."
      }
    }
  ],
  "usage": {
    "prompt_tokens": 25,
    "completion_tokens": 42,
    "total_tokens": 67
  }
}
```

The part we care about most is `choices[0].message.content` — that’s the LLM’s response text.

---
## 3. Your First LLM Call

Let’s build the request manually with `httpx.post()`. No wrappers, no helpers — just raw HTTP.

In [ ]:
# Build the request body
request_body = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "What is an AI agent? Explain in 2-3 sentences."}
    ],
    "temperature": 0.7,
}

print("Request body:")
print(json.dumps(request_body, indent=2))

In [ ]:
# Make the actual API call
try:
    response = httpx.post(
        API_URL,
        headers=HEADERS,
        json=request_body,
        timeout=60,
    )
    response.raise_for_status()
    data = response.json()

    # Print the raw response so you can see the full structure
    print("Raw response:")
    print(json.dumps(data, indent=2))

except httpx.HTTPStatusError as e:
    print(f"API error {e.response.status_code}: {e.response.text}")
except httpx.RequestError as e:
    print(f"Request failed: {e}")

In [ ]:
# Extract just the assistant's response text
try:
    assistant_message = data["choices"][0]["message"]["content"]
    print("LLM says:")
    print(assistant_message)
except (KeyError, IndexError, NameError) as e:
    print(f"Could not extract response: {e}")
    print("Make sure the previous cell ran successfully.")

That’s it — you just talked to an LLM. The pattern is always the same:

1. Build a list of messages
2. POST them to the API with your headers
3. Parse `choices[0].message.content` from the response

Everything else in this notebook (and in agent development generally) is variations on this theme.

---
## 4. Message Roles

The Chat Completions API uses three message roles:

| Role | Purpose | Example |
|------|---------|--------|
| `system` | Sets the LLM’s behavior, personality, or constraints | "You are a helpful coding assistant." |
| `user` | The human’s input | "How do I read a file in Python?" |
| `assistant` | The LLM’s previous responses (used in multi-turn) | "You can use `open()` to read files..." |

The **system prompt** is powerful — it shapes how the model responds to everything that follows. Let’s see the same question answered with two very different system prompts.

In [ ]:
question = "What is recursion?"

# --- System prompt 1: Explain like a teacher ---
messages_teacher = [
    {"role": "system", "content": "You are a patient computer science teacher. "
     "Explain concepts clearly using simple analogies. Keep answers under 3 sentences."},
    {"role": "user", "content": question},
]

try:
    response = httpx.post(
        API_URL, headers=HEADERS,
        json={"model": MODEL, "messages": messages_teacher, "temperature": 0.7},
        timeout=60,
    )
    response.raise_for_status()
    teacher_answer = response.json()["choices"][0]["message"]["content"]
    print("TEACHER:")
    print(teacher_answer)
except (httpx.HTTPStatusError, httpx.RequestError) as e:
    print(f"Request failed: {e}")

In [ ]:
time.sleep(1)  # Respect rate limits between calls

# --- System prompt 2: Explain like a stand-up comedian ---
messages_comedian = [
    {"role": "system", "content": "You are a stand-up comedian who explains tech concepts "
     "through jokes and humor. Keep answers under 3 sentences."},
    {"role": "user", "content": question},
]

try:
    response = httpx.post(
        API_URL, headers=HEADERS,
        json={"model": MODEL, "messages": messages_comedian, "temperature": 0.7},
        timeout=60,
    )
    response.raise_for_status()
    comedian_answer = response.json()["choices"][0]["message"]["content"]
    print("COMEDIAN:")
    print(comedian_answer)
except (httpx.HTTPStatusError, httpx.RequestError) as e:
    print(f"Request failed: {e}")

Same question, completely different style — all controlled by the system prompt. In agent development, the system prompt is where you define the agent’s personality, available tools, output format, and constraints.

---
## 5. Temperature and Sampling

**Temperature** controls how random the model’s output is:

| Temperature | Behavior | Use case |
|-------------|----------|----------|
| `0` | Deterministic — always picks the most likely token | Structured output, tool calls, code generation |
| `0.7` | Balanced — mostly coherent with some variety | General conversation, writing |
| `1.5` | Very creative — more surprising, sometimes incoherent | Brainstorming, creative writing |

Let’s see the difference. First, the same prompt at three different temperatures:

In [ ]:
prompt = "Invent a name for a friendly robot assistant."

for temp in [0, 0.7, 1.5]:
    try:
        response = httpx.post(
            API_URL, headers=HEADERS,
            json={
                "model": MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": temp,
                "max_tokens": 50,
            },
            timeout=60,
        )
        response.raise_for_status()
        text = response.json()["choices"][0]["message"]["content"]
        print(f"temp={temp}: {text.strip()[:100]}")
    except (httpx.HTTPStatusError, httpx.RequestError) as e:
        print(f"temp={temp}: Request failed: {e}")

    time.sleep(1)  # Respect rate limits

Now let’s demonstrate **consistency vs. variety**. We’ll send the same prompt 3 times at `temp=0` (should be nearly identical) and 3 times at `temp=1.0` (should vary).

In [ ]:
prompt = "Give me one word that describes the ocean."

print("--- temperature=0 (deterministic) ---")
for i in range(3):
    try:
        response = httpx.post(
            API_URL, headers=HEADERS,
            json={
                "model": MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
                "max_tokens": 20,
            },
            timeout=60,
        )
        response.raise_for_status()
        text = response.json()["choices"][0]["message"]["content"]
        print(f"  Run {i+1}: {text.strip()[:60]}")
    except (httpx.HTTPStatusError, httpx.RequestError) as e:
        print(f"  Run {i+1}: Request failed: {e}")
    time.sleep(1)

print()
print("--- temperature=1.0 (varied) ---")
for i in range(3):
    try:
        response = httpx.post(
            API_URL, headers=HEADERS,
            json={
                "model": MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 1.0,
                "max_tokens": 20,
            },
            timeout=60,
        )
        response.raise_for_status()
        text = response.json()["choices"][0]["message"]["content"]
        print(f"  Run {i+1}: {text.strip()[:60]}")
    except (httpx.HTTPStatusError, httpx.RequestError) as e:
        print(f"  Run {i+1}: Request failed: {e}")
    time.sleep(1)

Notice how `temp=0` gives the same (or very similar) answer every time, while `temp=1.0` produces different responses. For agent work, you’ll typically use low temperature (0–0.3) for tool calls and structured output, and moderate temperature (0.5–0.7) for natural conversation.

---
## 6. Multi-Turn Conversations

LLMs are **stateless** — they don’t remember previous calls. To have a conversation, **you** must send the full conversation history with every request. The model reads all the messages and generates the next response.

This is exactly how agents work: they maintain a list of messages and append to it on every turn.

In [ ]:
# Build a 3-turn conversation manually
conversation = [
    {"role": "system", "content": "You are a helpful assistant. Keep answers concise (1-2 sentences)."},
    {"role": "user", "content": "What is the capital of France?"},
]

# --- Turn 1 ---
try:
    response = httpx.post(
        API_URL, headers=HEADERS,
        json={"model": MODEL, "messages": conversation, "temperature": 0.3},
        timeout=60,
    )
    response.raise_for_status()
    reply_1 = response.json()["choices"][0]["message"]["content"]
    print(f"User: What is the capital of France?")
    print(f"Assistant: {reply_1}")
    
    # Add the assistant's reply to the conversation history
    conversation.append({"role": "assistant", "content": reply_1})
except (httpx.HTTPStatusError, httpx.RequestError) as e:
    print(f"Turn 1 failed: {e}")

In [ ]:
time.sleep(1)

# --- Turn 2 ---
conversation.append({"role": "user", "content": "What about Germany?"})

try:
    response = httpx.post(
        API_URL, headers=HEADERS,
        json={"model": MODEL, "messages": conversation, "temperature": 0.3},
        timeout=60,
    )
    response.raise_for_status()
    reply_2 = response.json()["choices"][0]["message"]["content"]
    print(f"User: What about Germany?")
    print(f"Assistant: {reply_2}")
    
    conversation.append({"role": "assistant", "content": reply_2})
except (httpx.HTTPStatusError, httpx.RequestError) as e:
    print(f"Turn 2 failed: {e}")

In [ ]:
time.sleep(1)

# --- Turn 3: A follow-up that requires context from earlier turns ---
conversation.append({"role": "user", "content": "Which of those two cities is older?"})

try:
    response = httpx.post(
        API_URL, headers=HEADERS,
        json={"model": MODEL, "messages": conversation, "temperature": 0.3},
        timeout=60,
    )
    response.raise_for_status()
    reply_3 = response.json()["choices"][0]["message"]["content"]
    print(f"User: Which of those two cities is older?")
    print(f"Assistant: {reply_3}")
    
    conversation.append({"role": "assistant", "content": reply_3})
except (httpx.HTTPStatusError, httpx.RequestError) as e:
    print(f"Turn 3 failed: {e}")

In [ ]:
# Let's see the full conversation history
print("Full conversation history:")
print(json.dumps(conversation, indent=2))

The key insight: the LLM answered "Which of those two cities is older?" correctly because it could see the entire conversation history, including the previous questions and answers about Paris and Berlin. Without that history, the model would have no idea what "those two cities" refers to.

This is why agents maintain a `messages` list — it’s the model’s only source of context.

---
## 7. Building a Reusable `chat()` Helper

We’ve been repeating the same boilerplate on every call: build headers, construct the body, call `httpx.post()`, check for errors, parse the response. Let’s extract that into a clean function.

In [ ]:
def chat(
    messages: list[dict],
    model: str = "meta-llama/llama-3.1-8b-instruct:free",
    temperature: float = 0.7,
    max_tokens: int | None = None,
) -> str:
    """Send messages to OpenRouter and return the assistant's response text.

    Args:
        messages: List of message dicts with 'role' and 'content' keys.
        model: OpenRouter model identifier.
        temperature: Sampling temperature (0 = deterministic, higher = more random).
        max_tokens: Optional max tokens in the response.

    Returns:
        The assistant's response text as a string.

    Raises:
        RuntimeError: If the API call fails or the response is malformed.
    """
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError(
            "OPENROUTER_API_KEY not found. "
            "Copy .env.example to .env and add your key."
        )

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }
    if max_tokens is not None:
        payload["max_tokens"] = max_tokens

    try:
        response = httpx.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=60,
        )
        response.raise_for_status()
    except httpx.HTTPStatusError as e:
        raise RuntimeError(
            f"OpenRouter API error {e.response.status_code}: {e.response.text}"
        ) from e
    except httpx.RequestError as e:
        raise RuntimeError(f"Request failed: {e}") from e

    data = response.json()

    try:
        return data["choices"][0]["message"]["content"]
    except (KeyError, IndexError) as e:
        raise RuntimeError(f"Unexpected response shape: {data}") from e


print("chat() function defined.")

In [ ]:
# Test it out — so much cleaner than the raw httpx calls!
try:
    reply = chat(
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Be concise."},
            {"role": "user", "content": "What are the three laws of robotics?"},
        ],
        temperature=0.3,
    )
    print(reply)
except RuntimeError as e:
    print(f"Error: {e}")

This exact `chat()` function lives in `utils/__init__.py` so you can import it in any notebook without redefining it. Let’s verify that works:

In [ ]:
import sys
sys.path.insert(0, "..")  # Add the repo root to Python's path

from utils import chat as shared_chat

# The shared version works identically
try:
    reply = shared_chat(
        messages=[{"role": "user", "content": "Say hello in 3 words."}],
        temperature=0,
    )
    print(f"Shared chat() says: {reply}")
except RuntimeError as e:
    print(f"Error: {e}")

From now on, in subsequent notebooks you’ll just do:

```python
import sys
sys.path.insert(0, "..")
from utils import chat
```

And you’re ready to talk to an LLM in one line.

---
## Putting It Together: Interactive Q&A Loop

Let’s combine everything into a mini-chatbot: an interactive loop where you type a question, the LLM responds, and the conversation continues with full history. Type `quit` to exit.

This is the simplest possible agent loop — no tools, no planning, just a human talking to an LLM in a loop.

In [ ]:
def interactive_chat():
    """Run an interactive multi-turn chat session with the LLM."""
    conversation = [
        {"role": "system", "content": "You are a helpful assistant. Keep answers concise."}
    ]

    print("Interactive Q&A (type 'quit' to exit)")
    print("=" * 40)

    while True:
        user_input = input("\nYou: ").strip()

        if not user_input:
            print("(empty input, try again)")
            continue

        if user_input.lower() == "quit":
            print("\nGoodbye!")
            break

        # Add user message to history
        conversation.append({"role": "user", "content": user_input})

        # Get the LLM's response
        try:
            reply = chat(messages=conversation, temperature=0.7)
            print(f"\nAssistant: {reply}")

            # Add assistant reply to history so the next turn has context
            conversation.append({"role": "assistant", "content": reply})
        except RuntimeError as e:
            print(f"\nError: {e}")
            # Remove the user message if the call failed
            conversation.pop()

    print(f"\nConversation had {len(conversation) - 1} messages (excluding system prompt).")


# Uncomment the line below to run the interactive chat:
# interactive_chat()

Uncomment `interactive_chat()` and run the cell to try it out. Notice how each response takes the full conversation history into account — the LLM remembers what you talked about because **you** are sending all previous messages on every turn.

This is the exact pattern we’ll extend into a real agent in notebook 02. The only difference: instead of always sending the reply straight to the user, the agent will check if the LLM wants to **call a tool** first.

### From teaching to production

This notebook kept things intentionally simple to focus on the core concepts. In a production application, you'd also want:

- **Retry with exponential backoff** — handle transient API errors (429, 503) gracefully instead of crashing
- **Streaming responses** — display tokens as they arrive instead of waiting for the full response
- **Token budget tracking** — monitor cumulative usage across calls to control costs

---
## Try It Yourself

Work through these exercises to reinforce what you’ve learned. Each one builds on a different section of this notebook.

### Exercise 1: Try a Different Model

Use the `chat()` function to send a message using a different free model from OpenRouter. Try one of these:

- `mistralai/mistral-7b-instruct:free`
- `google/gemma-2-9b-it:free`
- `qwen/qwen3-8b:free`

Ask the same question with two different models and compare the responses. Which model gives a better answer?

In [ ]:
# Exercise 1: Your code here
# Hint: chat() takes a `model` parameter
# Example: chat(messages=[...], model="mistralai/mistral-7b-instruct:free")


### Exercise 2: Pirate System Prompt

Write a system prompt that makes the LLM respond as a pirate. Ask it a normal question (like "How does Wi-Fi work?") and see how the persona affects the answer. Try to make it use pirate slang, say "arr", and refer to you as "matey".

In [ ]:
# Exercise 2: Your code here
# Hint: The system prompt is the first message in the messages list
# pirate_messages = [
#     {"role": "system", "content": "..."},
#     {"role": "user", "content": "How does Wi-Fi work?"},
# ]


### Exercise 3: Socratic Dialogue

Build a multi-turn conversation (at least 3 turns) where the LLM plays the role of Socrates. Use a system prompt that instructs it to answer questions with more questions (the Socratic method). Build the conversation manually like we did in Section 6 — append each assistant reply to the history before sending the next user message.

In [ ]:
# Exercise 3: Your code here
# Start with:
# socrates = [
#     {"role": "system", "content": "You are Socrates. ..."},
#     {"role": "user", "content": "What is justice?"},
# ]
# reply_1 = chat(messages=socrates)
# socrates.append({"role": "assistant", "content": reply_1})
# ... continue for 2-3 more turns


### Exercise 4: Token Counting

Modify the `chat()` function (or write a new version called `chat_with_usage()`) that also returns the token usage from the API response. The response includes a `usage` field:

```json
{"usage": {"prompt_tokens": 25, "completion_tokens": 42, "total_tokens": 67}}
```

Your function should return both the text and the usage dict. Make a few calls and print how many tokens each one used.

Hint: you’ll need to return a tuple `(content, usage)` or a dict instead of just a string.

In [ ]:
# Exercise 4: Your code here
# def chat_with_usage(messages, model=MODEL, temperature=0.7, max_tokens=None):
#     ... (similar to chat() but also extract and return data["usage"])
#     return content, usage
#
# text, usage = chat_with_usage([{"role": "user", "content": "Hello!"}])
# print(f"Response: {text}")
# print(f"Tokens used: {usage}")


---

**Next up:** [02. Basic Agent Loop](02_basic_agent_loop.ipynb) — extending this `chat()` function into a real agent that can decide when to call tools and when to respond to the user.